# Задача 5. Диффузия–реакция в 2D (химия, повышенная сложность)

Стационарная концентрация реагента в поперечном сечении катализатора $(x, y) \in [0,1]^2$.

PDE: $-\nabla^2 c + k c = f(x,y)$.

Аналитика: $c(x,y) = \sin(\pi x)\sin(\pi y)$, тогда $f = (2\pi^2 + k)\,c$.

На границе $c = 0$.

**Задание:** реализуйте `loss_data`, `loss_physics`, `loss_bc`.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch

here = Path.cwd().resolve()
ROOT = None
for p in [here, *here.parents]:
    if (p / "workshop" / "lib" / "workshop_common.py").exists():
        ROOT = p
        break
if ROOT is None:
    raise RuntimeError("Не найден корень PINN (ожидался workshop/lib/workshop_common.py)")

sys.path.insert(0, str(ROOT / "workshop"))
from lib.workshop_common import (
    DATA,
    MLP2D,
    laplacian,
    load_xyz_csv,
    plot_field_2d,
    set_seed,
)

set_seed(42)


## Данные и аналитическое решение


In [ ]:
k = 2.0

def analytical(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    return np.sin(np.pi * x) * np.sin(np.pi * y)

x_np, y_np, c_np = load_xyz_csv(DATA / "diffusion_reaction_2d.csv")
xy_data = torch.tensor(np.stack([x_np, y_np], axis=1))
c_data = torch.tensor(c_np).view(-1, 1)
model = MLP2D(n_hidden=64)
PLOT_TITLE = "PINN: диффузия–реакция 2D (−∇²c + k c = f)"
print(f"точек данных: {len(x_np)}")


## Функции потерь (эталон)


In [ ]:
def loss_data(model, xy, c):
    return torch.mean((model(xy) - c) ** 2)

def loss_physics(model, xy):
    xy = xy.clone().detach().requires_grad_(True)
    c = model(xy)
    lap = laplacian(c, xy)
    x = xy[:, 0:1]
    y = xy[:, 1:2]
    f = (2.0 * np.pi**2 + k) * torch.sin(np.pi * x) * torch.sin(np.pi * y)
    residual = -lap + k * c - f
    return torch.mean(residual ** 2)

def loss_bc(model, xy_bc):
    return torch.mean(model(xy_bc) ** 2)


## Обучение


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
lambda_data, lambda_phys, lambda_bc = 1.0, 1.0, 10.0
num_epochs = 4000
print_every = 500

# Collocation внутри области
n_phys = 400
xy_phys = torch.rand(n_phys, 2)

# Точки на границе квадрата [0, 1]²
n_bc = 100
t_edge = torch.linspace(0.0, 1.0, n_bc // 4)
zeros = torch.zeros_like(t_edge)
ones = torch.ones_like(t_edge)
xy_bc = torch.cat(
    [
        torch.stack([t_edge, zeros], dim=1),
        torch.stack([t_edge, ones], dim=1),
        torch.stack([zeros, t_edge], dim=1),
        torch.stack([ones, t_edge], dim=1),
    ],
    dim=0,
)

model.train()
for epoch in range(num_epochs):
    optimizer.zero_grad()
    l_d = loss_data(model, xy_data, c_data)
    l_p = loss_physics(model, xy_phys)
    l_b = loss_bc(model, xy_bc)
    loss = lambda_data * l_d + lambda_phys * l_p + lambda_bc * l_b
    loss.backward()
    optimizer.step()
    if (epoch + 1) % print_every == 0:
        print(
            f"epoch {epoch+1}/{num_epochs}  loss={loss.item():.5f}  "
            f"data={l_d.item():.5f}  phys={l_p.item():.5f}  bc={l_b.item():.5f}"
        )


## Сравнение полей


In [ ]:
model.eval()
n_plot = 80
xs = np.linspace(0.0, 1.0, n_plot, dtype=np.float32)
ys = np.linspace(0.0, 1.0, n_plot, dtype=np.float32)
x_grid, y_grid = np.meshgrid(xs, ys)
xy_plot = torch.tensor(np.stack([x_grid.ravel(), y_grid.ravel()], axis=1))
with torch.no_grad():
    c_pred = model(xy_plot).numpy().reshape(n_plot, n_plot)
c_true = analytical(x_grid, y_grid)
plot_field_2d(
    x_np, y_np, x_grid, y_grid, c_true, c_pred,
    title=PLOT_TITLE,
)
